# Stringify unknown values in data series

Issue [#184](https://github.com/JetBrains/lets-plot-kotlin/issues/184).

**Before**: HTML export and notebook display failed with `Can't serialize object A(a=1)` when a data series contained a custom class. PNG export already worked.

**Fix**: unknown scalars inside list-like data values are converted with `toString()` during `plot.toSpec()`. This covers `data`, inline collection mappings, and scale `limits` / `breaks`.

**Section 1** checks the original `ggsave` failure path. Sections 2–11 cover the main standardizer paths.

In [ ]:
%useLatestDescriptors
%use lets-plot

In [ ]:
LetsPlot.getInfo()

## 1. End-to-end: `ggsave` to HTML

Reproduces issue #184 and exercises HTML and SVG export. Before the fix, HTML export failed in `PlotHtmlExport.buildHtmlFromRawSpecs`.

The cell also checks the saved HTML for `A(a=...)`, confirming the embedded spec contains stringified values.

Sections 2–11 cover individual standardizer paths.

In [ ]:
import java.io.File
import java.nio.file.Files

// Issue #184 reproduction.
data class A(val a: Int)

val pIssue = letsPlot() + geomPoint(size = 6) {
    x     = listOf(A(1), A(2))
    y     = listOf(1, 2)
    color = listOf(A(1), A(2))
}

val outDir = Files.createTempDirectory("issue184_").toString()

// This used to throw: Can't serialize object A(a=1).
val htmlPath = ggsave(pIssue, "plot.html", path = outDir)
val svgPath  = ggsave(pIssue, "plot.svg",  path = outDir)

println("HTML: $htmlPath")
println("SVG:  $svgPath")

// The embedded HTML spec should contain stringified A(a=...) values.
val htmlText = File(htmlPath).readText()
val matches = Regex("A\\(a=\\d+\\)").findAll(htmlText).map { it.value }.toSortedSet()
println("Stringified values found in HTML: $matches")

pIssue

## 2. Data class in `ggplot(data = ...)`

A custom `data class` inside the plot `data` map now uses its default `toString()`.

In [ ]:
data class Category(val id: Int, val name: String)

val data1 = mapOf(
    "cat" to listOf(Category(1, "alpha"), Category(2, "beta"), Category(3, "gamma")),
    "y"   to listOf(3.0, 1.5, 4.2)
)

val p1 = letsPlot(data1) + geomBar(stat = Stat.identity) { x = "cat"; y = "y"; fill = "cat" }
println(p1.toSpec()["data"])
p1

## 3. Custom `toString()` is honored

The object's `toString()` value is used in axes, legends, and tooltips.

In [ ]:
class Money(val amount: Int, val currency: String) {
    override fun toString() = "$amount $currency"
}

val data2 = mapOf(
    "price" to listOf(Money(10, "USD"), Money(25, "EUR"), Money(7, "GBP")),
    "sold"  to listOf(120, 80, 200)
)

val p2 = letsPlot(data2) + geomPoint(size = 6) { x = "price"; y = "sold"; color = "price" }
println(p2.toSpec()["data"])
p2

## 4. Inline collection mapping inside a layer

`geomPoint { x = listOf(...) }` follows the same path.

In [ ]:
data class Region(val code: String) { override fun toString() = code }

val p3 = ggplot() + geomPoint(size = 6) {
    x     = listOf(Region("NA"), Region("EU"), Region("APAC"))
    y     = listOf(1, 2, 3)
    color = listOf(Region("NA"), Region("EU"), Region("APAC"))
}
println((p3.toSpec()["layers"] as List<*>).first())
p3

## 5. Scale `limits` / `breaks` with unknown scalars

Limits and breaks use the same list standardizer.

In [ ]:
data class Tag(val v: String) { override fun toString() = "#$v" }

val data4 = mapOf(
    "tag"   to listOf("#a", "#b", "#c", "#d"),
    "count" to listOf(4, 7, 2, 5)
)

val p4 = letsPlot(data4) +
    geomBar(stat = Stat.identity) { x = "tag"; y = "count" } +
    scaleXDiscrete(limits = listOf(Tag("a"), Tag("b"), Tag("c"), Tag("d")))
println(p4.toSpec()["scales"])
p4

## 6. Mixed scalar types in the same series

Known values stay typed; unknown scalars become strings.

In [ ]:
data class Marker(val n: Int) { override fun toString() = "M$n" }

val data5 = mapOf(
    "label" to listOf("plain", 42, Marker(1), Marker(2), null),
    "y"     to listOf(1, 2, 3, 4, 5)
)

val p5 = letsPlot(data5) + geomPoint(size = 6) { x = "label"; y = "y"; color = "label" }
println(p5.toSpec()["data"])
p5

## 7. Enums

Enums are handled by the strict standardizer and become their constant names.

In [ ]:
enum class Severity { LOW, MEDIUM, HIGH }

val data6 = mapOf(
    "sev"   to listOf(Severity.LOW, Severity.MEDIUM, Severity.HIGH, Severity.MEDIUM, Severity.LOW),
    "count" to listOf(12, 7, 3, 9, 15)
)

val p6 = letsPlot(data6) + geomBar(stat = Stat.identity) { x = "sev"; y = "count"; fill = "sev" }
println(p6.toSpec()["data"])
p6

## 8. `Color` values in a series

`Color` already becomes a hex string in the strict standardizer. `scaleFillIdentity()` renders those hex values as colors.

In [ ]:
import org.jetbrains.letsPlot.commons.values.Color

val data7 = mapOf(
    "label" to listOf("a", "b", "c"),
    "y"     to listOf(1.0, 2.0, 3.0),
    "col"   to listOf(Color.RED, Color.GREEN, Color.BLUE)
)

val p7 = letsPlot(data7) +
    geomBar(stat = Stat.identity) { x = "label"; y = "y"; fill = "col" } +
    scaleFillIdentity()
println(p7.toSpec()["data"])
p7

## 9. Temporal values mixed with unknowns

`LocalDate` becomes epoch milliseconds; custom `Episode` values use `toString()` in the same series.

In [ ]:
import kotlinx.datetime.LocalDate

data class Episode(val n: Int) { override fun toString() = "E$n" }

val data8 = mapOf(
    "when"  to listOf(LocalDate(2026, 1, 1), LocalDate(2026, 2, 1), LocalDate(2026, 3, 1)),
    "tag"   to listOf(Episode(1), Episode(2), Episode(3)),
    "value" to listOf(10, 20, 15)
)

val p8 = letsPlot(data8) +
    geomPoint(size = 6) { x = "when"; y = "value"; color = "tag" } +
    scaleXDateTime()
println(p8.toSpec()["data"])
p8

## 10. Non-`List` iterables: `Sequence`, `Array`, `Pair`

`Sequence`, `Array`, primitive arrays, and `Pair` are list-like too, so their unknown scalar elements are stringified.

Nested-list recursion is covered by `StandardizingTest.unknown_scalar_values_nested_in_a_series_list_are_stringified`.

In [ ]:
data class Item(val n: Int) { override fun toString() = "Item$n" }

fun mappingOf(p: org.jetbrains.letsPlot.intern.Plot): Map<*, *> =
    ((p.toSpec()["layers"] as List<*>).first() as Map<*, *>)["mapping"] as Map<*, *>

// Sequence
val p9seq = ggplot() + geomPoint(size = 6) {
    x = sequenceOf(Item(1), Item(2), Item(3))
    y = listOf(1, 2, 3)
}
println("Sequence: " + mappingOf(p9seq))

// Array
val p9arr = ggplot() + geomPoint(size = 6) {
    x = arrayOf(Item(4), Item(5), Item(6))
    y = listOf(1, 2, 3)
}
println("Array:    " + mappingOf(p9arr))

// Pair
val p9pair = ggplot() + geomPoint(size = 6) {
    x = Item(7) to Item(8)
    y = listOf(1, 2)
}
println("Pair:     " + mappingOf(p9pair))

p9seq

## 11. Strict boundary

Unknown scalars passed as plain option values are not stringified. This protects sentinel objects like `MappingMeta`.

Reflection shows the boundary: plain `Holder` is preserved, but `listOf(Holder)` is stringified.

In [ ]:
data class Holder(val n: Int)

val standardizingClass = Class.forName("org.jetbrains.letsPlot.intern.standardizing.Standardizing")
val standardizingInstance = standardizingClass.getField("INSTANCE").get(null)
val standardizeValue = standardizingClass
    .getDeclaredMethod("standardizeValue", Any::class.java)
    .apply { isAccessible = true }

val holder = Holder(7)

// Plain option value: same instance returned.
val strict = standardizeValue.invoke(standardizingInstance, holder)
println("Plain:   in=$holder  out=$strict  (${strict!!::class.simpleName})  same? ${strict === holder}")

// Inside a list: stringified. Print the element type to make that visible.
val asList = standardizeValue.invoke(standardizingInstance, listOf(holder)) as List<*>
println("In list: in=[${holder}] (${holder::class.simpleName})  out=$asList  (element: ${asList[0]!!::class.simpleName})")